# KB Card 파일 불러오기 (정산월 날짜 보정 적용)

## 0. 초기 세팅
- 파일은 대금명세서로 조회하면 뜨는 별도의 창에서 이용내역만 다운로드

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
if str(project_root / "src") not in sys.path:
    sys.path.append(str(project_root / "src"))

import pandas as pd
import numpy as np
from ledgerly.expenditure import (
    preprocess_kbcard_data,
    map_kb_card_df_to_expenditure,
    map_category,
    insert_expenditure_data,
)
from ledgerly.expenditure.config import kbcard_file_config

## 1. 데이터 파일 조회 및 전처리

In [2]:
target_yymm = "2603"
target_filename = "202603_kbcard.xlsx"
target_file = kbcard_file_config["data_dir"] / target_yymm / target_filename

print(f"Target file: {target_file}")

raw_df = pd.read_excel(target_file, header=None)
data_df = preprocess_kbcard_data(raw_df)

print(f"전처리 완료: {len(data_df)}건")
data_df.head()

Target file: E:\workspaces\ledgerly\data\raw\2603\202603_kbcard.xlsx
전처리 완료: 97건


e:\workspaces\ledgerly\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,dummy_0,이용일자,이용카드,구분,이용하신 가맹점,dummy_5,이용금액,할부개월,회차,원금,수수료_이자,회차2,원금2,적립예정포인트,실제결제금액,이용금액_원본,비고
3,NaN,2026-03-01,마스터007,일시불,파리바게트김포자이점,,6300,,,6300,,,,,6300,6300,None
4,NaN,2026-03-01,비자056,일시불,메가MGC커피김포금빛초점,,3500,,,3500,,,,58,3500,3500,None
5,NaN,2026-03-01,마스터007,일시불,메가MGC커피김포금빛초점,,1900,,,1900,,,,,1900,2000,None
7,NaN,2026-03-02,마스터007,일시불,메가MGC커피김포금빛초점,,3135,,,3135,,,,,3135,3300,None
9,NaN,2026-03-02,마스터007,일시불,씨유(CU)김포센트럴자이점,,4085,,,4085,,,,,4085,4300,None


## 2. DB 데이터로 매핑 및 날짜 보정

In [3]:
# 정산 기준일 설정 (예: 2026-03-01)
statement_date = f"20{target_yymm[:2]}-{target_yymm[2:]}-01"

expenditure_df = map_kb_card_df_to_expenditure(data_df, statement_date=statement_date)

# 중복 데이터 제거
before_count = len(expenditure_df)
expenditure_df = expenditure_df.drop_duplicates(subset=["source_uid"])
after_count = len(expenditure_df)

if before_count != after_count:
    print(f"중복 데이터 {before_count - after_count}건을 제거하였습니다.")

expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
expenditure_df["category"] = expenditure_df["category"].fillna("미분류")

print(f"매핑 완료: {len(expenditure_df)}건 (정산월: {statement_date})")
expenditure_df.head(20)

매핑 완료: 97건 (정산월: 2026-03-01)


,used_at,merchant_name,installment_type,amount,source_uid,memo,payment_type,payment_provider,remaining_amount,category
3,2026-03-01,파리바게트김포자이점,single,6300,kb_20260301_파리바게트김포자이점_6300_1,None,card,kb,0,간식
4,2026-03-01,메가MGC커피김포금빛초점,single,3500,kb_20260301_메가MGC커피김포금빛초점_3500_1,None,card,kb,0,간식
5,2026-03-01,메가MGC커피김포금빛초점,single,1900,kb_20260301_메가MGC커피김포금빛초점_1900_1,None,card,kb,0,간식
7,2026-03-02,메가MGC커피김포금빛초점,single,3135,kb_20260302_메가MGC커피김포금빛초점_3135_1,None,card,kb,0,간식
9,2026-03-02,씨유(CU)김포센트럴자이점,single,4085,kb_20260302_씨유CU김포센트럴자이점_4085_1,None,card,kb,0,편의
11,2026-03-03,커피빈김포장기DT점,single,6460,kb_20260303_커피빈김포장기DT점_6460_1,None,card,kb,0,기타
13,2026-03-03,웨이브(푹TV)자동-웨이브(푹TV),single,10900,kb_20260303_웨이브푹TV자동웨이브푹TV_10900_1,None,card,kb,0,문화
14,2026-03-04,김포감정주유소,single,56000,kb_20260304_김포감정주유소_56000_1,None,card,kb,0,교통
15,2026-03-04,커피빈김포장기DT점,single,6175,kb_20260304_커피빈김포장기DT점_6175_1,None,card,kb,0,기타
17,2026-03-05,커피빈김포장기DT점,single,6650,kb_20260305_커피빈김포장기DT점_6650_1,None,card,kb,0,기타


## 3. DB 저장

In [4]:
if not expenditure_df.empty:
    insert_expenditure_data(expenditure_df)
    print(f"{len(expenditure_df)}개의 데이터가 성공적으로 저장되었습니다.")
else:
    print("저장할 데이터가 없습니다.")

97개의 데이터가 성공적으로 저장되었습니다.
